# Tercile Hit Rate Analysis of Climate Models

Notes and Todos at the end of the document

In [45]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [46]:
# import necessary libraries
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns

In [47]:
import os
from collections import defaultdict

# define list missing months function
def list_missing_months(file_path):
  """This function takes a model file path, such as the GFDL model, and lists
  the missing months in the file names.

  For example, if the year 1997 had the missing month of September, the
  function will print '1997 missing Sep'.

  Assumptions
  -----------
  Assumes that the path given to the function leads to a folder that has files
  with naming convention similar to 'prec.GFDL-SPEAR.1997.mon_Aug.nc'

  This function slices the last part of the file name to get the year and month
  information. It ignores the "front" part of the file name. So, in theory, as long
  as the format is 'some_variable.some_model_name.YYYY.mon_MMM.nc', this function will
  be able to check all missing months.

  Assumes that for any given year, months with that data exist. For example, if it is
  November 2025 currently, and I have all available data, running this function will return
  '2025 missing Dec'

  Dependencies
  ------------
  Make sure to import the os module and the defaultdict object using these import statements.
  These are native to python and require no additional installation.

  import os
  from collections import defaultdict
  """

  # use os module to list all file names
  file_list = os.listdir(file_path)

  # Extract year and month info from filenames
  # for the GFDL model, the file names are in the format of 'prec.GFDL-SPEAR.1997.mon_Aug.nc'
  # Thus, the year and month are indexed by file_name[16:20] and file_name[25:28] respectively

  # define year_month dictionary
  year_month_dict = defaultdict(set)

  for file in file_list:
      if len(file) >= 28:  # Ensure the filename is long enough
          year = file[-15:-11]  # Extract year (e.g., "2024")
          month = file[-6:-3]  # Extract month (e.g., "Sep")
          year_month_dict[year].add(month)

  # Define the complete set of months
  full_month_set = {'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'}

  # Check for missing months
  for year, months in year_month_dict.items():
      missing_months = full_month_set - months  # Find missing months
      if missing_months:
          missing_str = ', '.join(sorted(missing_months))
          print(f"{year} missing {missing_str}")

# Example usage

# Define the file path of GFDL model
file_path = '/content/drive/My Drive/capstone_data/NMME/GFDL-SPEAR/prec'

# call the function
list_missing_months(file_path)

2025 missing Apr, Aug, Dec, Feb, Jul, Jun, Mar, May, Nov, Oct, Sep
2020 missing Apr, Aug, Feb, Jan, Jul, Jun, Mar, May, Nov, Oct, Sep
1997 missing Sep


In [48]:
# Set index to be reindex to

# -90 to 90, with 0.5 degree resolution latitude
# -180 to 180 with 0.5 degree resolution longitude
new_lat = np.arange(-90, 90, 0.5)
new_lon = np.arange(-180, 180, 0.5)

In [49]:
# open chirps dataset
chirps = xr.open_dataset('/content/drive/My Drive/capstone_data/CHIRPS/chirps-v2.0.monthly.nc')

In [50]:
# open GFDL dataset
GFDL = xr.open_mfdataset('/content/drive/My Drive/capstone_data/NMME/GFDL-SPEAR/prec/*.nc', parallel=True)

# regrid to have same latitude and longitude as chirps
GFDL = GFDL.assign_coords(X=(((GFDL.X + 180) % 360) - 180)).sortby(['X'])

# reindex to 0.5 degree resolution, using the latitude and longitude previously defined (chirps)
GFDL = GFDL.reindex(X=new_lon, Y=new_lat).ffill('X').ffill('Y')

# Attempt to run tercile hit rate for South Sudan Region GFDL

Only use years 1993 to 2020

In [51]:
# Subset the South Sudan Region GFDL
GFDL_south_sudan = (GFDL.sel(Y=slice(3.5, 12.5), X=slice(25, 35))
                       .rename({'Y': 'latitude', 'X': 'longitude', 'prec': 'predicted_precip', 'S': 'date_of_prediction', 'L': 'lead_time'}))

# Interpolate and subset CHIRPS to match spatial resolution of GFDL
chirps_south_sudan = chirps.sel(latitude=slice(3,13), longitude=slice(24.5,35.5)).interp_like(GFDL_south_sudan, method='nearest')

In [52]:
# Took 5 mins to run on colab
# Calculate realized dates for GFDL
GFDL_south_sudan_df = GFDL_south_sudan.to_dataframe().reset_index()
GFDL_south_sudan_df['realization_time'] = GFDL_south_sudan_df['date_of_prediction'] + (GFDL_south_sudan_df['lead_time'] * 30).astype('timedelta64[D]')
GFDL_south_sudan_df['month'] = GFDL_south_sudan_df['realization_time'].dt.month
GFDL_south_sudan_df['year'] = GFDL_south_sudan_df['realization_time'].dt.year

# Convert CHIRPS to dataframe
chirps_south_sudan_df = chirps_south_sudan.to_dataframe().reset_index()
chirps_south_sudan_df['month'] = chirps_south_sudan_df['time'].dt.month
chirps_south_sudan_df['year'] = chirps_south_sudan_df['time'].dt.year

# Define a tercile category assignment function

In [88]:
def assign_tercile_category(dataframe, variable_name, dataset_name):
  """Takes a dataframe and a variable name, and returns a dataframe with a column
  for the tercile category of the variable.

  Assumptions:
  Desired model has been converted to dataframe format
  You know the variable of interest to compute terciles for, such as 'precip'

  Usage:
  GFDL_tercile_category_df = assign_tercile_category(GFDL_south_sudan_df, 'predicted_precip', 'GFDL')

  The dataset_name argument makes the dataframe more readable and convenient for analysis

  Returns:
  A dataframe with a column for the tercile category of the variable.
  """

  # Compute tercile thresholds
  lower_tercile = dataframe[variable_name].quantile(0.33)
  upper_tercile = dataframe[variable_name].quantile(0.66)

  # Classify precipitation using vectorized NumPy operations
  conditions = [
        dataframe[variable_name] < lower_tercile,
        dataframe[variable_name] > upper_tercile
    ]
  choices = ["Low", "High"]

  # Assign categories (default is "Medium")
  dataframe[f"tercile_category_{dataset_name}"] = np.select(conditions, choices, default="Medium")

  return dataframe

In [83]:
# There is lots of missing data when I select the region
GFDL_south_sudan_df.isna().sum()

,0
lead_time,0
latitude,0
M,0
date_of_prediction,0
longitude,0
predicted_precip,6822900
realization_time,0
month,0
year,0


In [81]:
# ?? this subsetting doesn't work, but it works for chirps
# it bugs out a lot

# subset where year is 1993 to 2020
# GFDL_south_sudan_df_1993_2020 = GFDL_south_sudan_df[(GFDL_south_sudan_df['year'] >= 1993) & (GFDL_south_sudan_df['year'] <= 2020)]

# GFDL_south_sudan_df_1993_2020



# below code works in theory if above is fixed

# take the terciles and assign the classification
GFDL_south_sudan_df_terciles = assign_tercile_category(GFDL_south_sudan_df_1993_2020, 'precip', 'GFDL')

# View the tercile classification dataframe
GFDL_south_sudan_df_terciles

,lead_time,latitude,M,date_of_prediction,longitude,predicted_precip,realization_time,month,year
20826745,11.5,12.5,30.0,1998-12-01,33.0,NaN,1999-11-11,11,1999
20826746,11.5,12.5,30.0,1998-12-01,33.5,NaN,1999-11-11,11,1999
20826747,11.5,12.5,30.0,1998-12-01,34.0,NaN,1999-11-11,11,1999
20826748,11.5,12.5,30.0,1998-12-01,34.5,NaN,1999-11-11,11,1999
20826749,11.5,12.5,30.0,1998-12-01,35.0,NaN,1999-11-11,11,1999


In [89]:
# subset where year is 1993 to 2020
chirps_south_sudan_df_1993_2020 = chirps_south_sudan_df[(chirps_south_sudan_df['year'] >= 1993) & (chirps_south_sudan_df['year'] <= 2020)]

# take the terciles and assign the classification
chirps_south_sudan_df_terciles = assign_tercile_category(chirps_south_sudan_df_1993_2020, 'precip', 'chirps')

# View the tercile classification dataframe
chirps_south_sudan_df_terciles


<ipython-input-88-fd78a5254b1e>:30: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataframe[f"tercile_category_{dataset_name}"] = np.select(conditions, choices, default="Medium")


,time,latitude,longitude,precip,month,year,tercile_category_chirps
57456,1993-01-01,3.5,25.0,21.016228,1,1993,Medium
57457,1993-01-01,3.5,25.5,26.995951,1,1993,Medium
57458,1993-01-01,3.5,26.0,32.001976,1,1993,Medium
57459,1993-01-01,3.5,26.5,58.396255,1,1993,Medium
57460,1993-01-01,3.5,27.0,35.482803,1,1993,Medium
...,...,...,...,...,...,...,...
191515,2020-12-01,12.5,33.0,0.021808,12,2020,Low
191516,2020-12-01,12.5,33.5,0.079275,12,2020,Low
191517,2020-12-01,12.5,34.0,0.087982,12,2020,Low
191518,2020-12-01,12.5,34.5,0.075248,12,2020,Low


In [ ]:
# compute the percentage of times where GFDL and CHIRPS agree on tercile classification

# merge on year and month
tercile_merged_south_sudan_GFDL = pd.merge(GFDL_south_sudan_df_terciles, chirps_south_sudan_df_terciles, on=['year', 'month'])

# compute agreement rate
df["agreement_rate"] = df["tercile_category_chirps"] == df["tercile_category_GFDL"]

# Calculate agreement rate per year
agreement_per_year = df.groupby("year")["agreement_rate"].mean() * 100

# Print results
agreement_per_year


In [93]:
# example of what final output would look like

# Example merged dataframe (select columns)
df = pd.DataFrame({
    "year": [2000, 2000, 2001, 2001, 2002, 2002],
    "month": [1, 2, 1, 2, 1, 2],
    "tercile_category_chirps": ["Low", "Medium", "High", "Low", "Medium", "High"],
    "tercile_category_gfdl": ["Low", "High", "High", "Medium", "Medium", "High"]
})

# see if there is agreement
df["agreement_rate"] = df["tercile_category_chirps"] == df["tercile_category_gfdl"]

# Calculate agreement rate per year
agreement_per_year = df.groupby("year")["agreement_rate"].mean() * 100

# Print results
agreement_per_year

,agreement_rate
year,
2000,50.0
2001,50.0
2002,100.0


In [94]:
# NEED TO IMPLEMENT ENSEMBLE MEANS
# CAN CALCULATE MONTHLY OR YEARLY PRECIP MEANS AND THEN TAKE TERCILES DEPENDING ON WHAT'S DESIRED
# FIGURE OUT WHAT'S HAPPENING WITH GFDL SUBSETTING CREATING NANS
# 1997 SEPTEMBER FOR GFDL IS MISSING, NOT ON FTP